In [32]:
# Django starter

from pars_django_loader import DjangoLoader
DjangoLoader.auto()

Django has started...


In [33]:
from counter.models import CounterEntry, Counter, CounterSplitType
from django.db.models import Sum
import calendar


In [3]:
test_counter = Counter.objects.first()

In [ ]:
entryies = CounterEntry.objects.filter(counter=test_counter)

In [ ]:
for entry in entryies:
    print(entry)

In [ ]:
from django.db.models.functions import TruncDate

In [ ]:
entryies = entryies.annotate(date=TruncDate('timestamp'))

In [ ]:
from django.db.models.functions import TruncDate
from django.db.models.functions import TruncDay, TruncWeek, TruncMonth
from django.db.models import Count
from counter.models import CounterSplitType


def split(split_type: CounterSplitType = CounterSplitType.DAILY):
    trunc_function = CounterSplitType.get_trunc_function(
        split_type
    )

    entryies = CounterEntry.objects.filter(counter=test_counter)
    entryies = entryies.annotate(date=trunc_function('timestamp'))
    
    grouped_entries = entryies.values('date').annotate(count=Count('id')).order_by('date')
    
    for entry in grouped_entries:
        print(entry['date'], entry['count'])


In [ ]:
split(CounterSplitType.DAILY)

In [ ]:
split(CounterSplitType.WEEKLY)

In [ ]:
split(CounterSplitType.MONTHLY)

In [ ]:
entryies = entryies.values('date', 'counter_id', 'counter__name')

In [ ]:
entryies = entryies.annotate(total=Sum('value'))

In [ ]:
entryies = entryies.order_by('-date')

In [ ]:
# custom

In [ ]:
from django.db.models.functions import Coalesce
from django.db.models import Sum, Value, Q
from datetime import date


In [ ]:
counter = Counter.objects.all()
today = date.today()


In [ ]:
counter.annotate(
    count=Coalesce(Sum('entries__value'), Value(0)),
    today_count=Coalesce(Sum('entries__value', filter=Q(entries__timestamp__date=today)), Value(0))
)

In [32]:
from datetime import datetime, timedelta

today = datetime.today()

start_of_week = today - timedelta(days=today.weekday())
end_of_week = start_of_week + timedelta(days=6)

print("Haftanın ilk günü:", start_of_week.date())
print("Haftanın son günü:", end_of_week.date())


Haftanın ilk günü: 2025-08-04
Haftanın son günü: 2025-08-10


In [38]:
from datetime import datetime, time

month_start = today.replace(day=1)
month_start = datetime.combine(month_start.date(), time.min)

last_day = calendar.monthrange(today.year, today.month)[1]
month_end = today.replace(day=last_day)
month_end = datetime.combine(month_end.date(), time.max)

print("Baş:", month_start)
print("Son:", month_end)


Baş: 2025-08-01 00:00:00
Son: 2025-08-31 23:59:59.999999


# serializer week test

In [4]:
from rest_framework import serializers
from counter.models import CounterEntry, Counter, CounterSplitType
from datetime import datetime, timedelta, date
from django.db.models import Sum, Value, Q
from django.db.models.functions import Coalesce

In [26]:
today = datetime.today()

start_of_week = today - timedelta(days=today.weekday())
end_of_week = start_of_week + timedelta(days=6)
case_query = Q(entries__timestamp__date__range=( start_of_week, end_of_week ))


counters = Counter.objects.all()
for counter in counters.annotate(today_count=Coalesce(Sum('entries__value', filter=case_query), Value(0))):
    print(counter.today_count)

2
0
0
0
0
1
1
0
0
0
1
0
1
1
18
0
0


In [36]:
today = datetime.today()

month_start = today.replace(day=1)
month_start = datetime.combine(month_start.date(), time.min)

last_day = calendar.monthrange(today.year, today.month)[1]
month_end = today.replace(day=last_day)
month_end = datetime.combine(month_end.date(), time.max)


NameError: name 'time' is not defined

In [28]:
month_start

datetime.datetime(2025, 8, 1, 19, 51, 46, 537211)

In [29]:
month_end

datetime.datetime(2025, 8, 31, 19, 51, 46, 537211)

In [31]:
case_query = Q(entries__timestamp__date__range=( month_start, month_end ))

counters = Counter.objects.all()
for counter in counters.annotate(today_count=Coalesce(Sum('entries__value', filter=case_query), Value(0))):
    print(counter.today_count)

2
0
0
0
0
8
1
0
0
0
1
0
-15
1
-10
-8
0


In [39]:
start = datetime.combine(today, time.min)
end = datetime.combine(today, time.max)

In [50]:
if start and end:
    case_query = Q(entries__timestamp__range=(start, end))
else:
    case_query = Q()

In [51]:
case_query

<Q: (AND: ('entries__timestamp__range', (datetime.datetime(2025, 8, 4, 0, 0), datetime.datetime(2025, 8, 4, 23, 59, 59, 999999))))>

In [56]:
interval_count = test_counter.entries.aggregate(
    interval_count=Coalesce(
        Sum('value', filter=Q(timestamp__range=(start, end))),
        Value(0)
    )
)['interval_count']

In [57]:
interval_count

18